In [ ]:
import gc
import pandas as pd
import numpy as np
import pickle
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import SGDRegressor
from sklearn.linear_model import HuberRegressor
from tqdm import tqdm
import statsmodels.api as sm
from sklearn.metrics import r2_score
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, Dataset
import argparse
from torch.autograd import Variable
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import r2_score
from sklearn.preprocessing import StandardScaler

In [ ]:
with open("data/characteristics_with_ind_interaction_return.pkl", "rb") as f:
    data = pickle.load(f)

In [ ]:
import os

# 使用 free -h 命令查看内存使用情况
memory_info = os.popen('free -h').read()
print(memory_info)

# NN

In [ ]:
class NN4(nn.Module):
    def __init__(self, num_inputs, num_hidden, num_output):
        super(NN4, self).__init__()
        
        # 定义网络结构
        self.model = nn.Sequential(
            nn.Linear(num_inputs, num_hidden),
            nn.ReLU(),
            nn.Linear(num_hidden, num_hidden),
            nn.ReLU(),
            nn.Linear(num_hidden,num_hidden),
            nn.ReLU(),
            nn.Linear(num_hidden,num_output)
        )
        
        # 初始化权重
        self._initialize_weights()

    def _initialize_weights(self):
        for m in self.model:
            if isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, std=0.01)
    
    def forward(self, x):
        return self.model(x)

In [ ]:
class NN5(nn.Module):
    def __init__(self, num_inputs, num_hidden, num_output):
        super(NN5, self).__init__()
        
        # 定义网络结构
        self.model = nn.Sequential(
            nn.Linear(num_inputs, num_hidden),
            nn.ReLU(),
            nn.Linear(num_hidden, num_hidden),
            nn.ReLU(),
            nn.Linear(num_hidden,num_hidden),
            nn.ReLU(),
            nn.Linear(num_hidden,num_hidden),
            nn.ReLU(),
            nn.Linear(num_hidden,num_output)
        )
        
        # 初始化权重
        self._initialize_weights()

    def _initialize_weights(self):
        for m in self.model:
            if isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, std=0.01)
    
    def forward(self, x):
        return self.model(x)

# NN2

In [ ]:
num_inputs=94
num_output=1
num_hidden=256
batch_size=256
num_epochs=60
criterion = nn.MSELoss()

class NN2(nn.Module):
    def __init__(self, num_inputs, num_hidden, num_output):
        super(NN2, self).__init__()
        # 定义网络结构
        self.model = nn.Sequential(
            nn.Linear(num_inputs, num_hidden),
            nn.ReLU(),
            nn.Linear(num_hidden, num_output)
        )
        
        # 初始化权重
        self._initialize_weights()

    def _initialize_weights(self):
        for m in self.model:
            if isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, std=0.01)
    
    def forward(self, x):
        return self.model(x)

In [ ]:
class Net3(nn.Module):
    def __init__(self):
        self.name = 'Net3'
        super(Net3, self).__init__()
        self.fc1 = nn.Linear(94,16)
        self.fc2 = nn.Linear(16,8)
        self.fc3 = nn.Linear(8,4)
        self.fc4 = nn.Linear(4,1)
        
    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x)) 
        x = F.relu(self.fc3(x)) 
        x = self.fc4(x)
        return x

In [ ]:
start_year_train = 1957
start_year_test = 1987
end_year_test = 2016

# Convert the 'Date' column to datetime if it's not already
data['DATE'] = pd.to_datetime(data['DATE'], format='%Y%m')

pred_y_list = []
true_y_list = []

# Loop over the years
for year in tqdm(range(start_year_test, end_year_test + 1), desc='Processing', unit='item'):
    # Define the start and end dates for training and test
    start_date_train = pd.Timestamp(year=start_year_train, month=1, day=1)
    end_date_train = pd.Timestamp(year=year - 13, month=12, day=31)

    start_date_val = pd.Timestamp(year=year - 12, month=1, day=1)
    end_date_val = pd.Timestamp(year=year - 1, month=12, day=31)

    start_date_test = pd.Timestamp(year=year, month=1, day=1)
    end_date_test = pd.Timestamp(year=year, month=12, day=31)

    print(f'start_date_train: {start_date_train.strftime("%Y-%m")}')
    print(f'end_date_train: {end_date_train.strftime("%Y-%m")}')
    print(f'start_date_val: {start_date_val.strftime("%Y-%m")}')
    print(f'end_date_val: {end_date_val.strftime("%Y-%m")}')
    print(f'start_date_test: {start_date_test.strftime("%Y-%m")}')
    print(f'end_date_test: {end_date_test.strftime("%Y-%m")}')
    
    # Select the data for training and test
    train_data = data[(data['DATE'] >= start_date_train) & (data['DATE'] <= end_date_val)]
    test_data = data[(data['DATE'] >= start_date_test) & (data['DATE'] <= end_date_test)]
    
    scaler = StandardScaler()
    X_train = scaler.fit_transform(train_data.drop(['permno', 'DATE', 'RET'], axis=1).values)
    y_train = train_data['RET'].values
    X_test = scaler.transform(test_data.drop(['permno', 'DATE', 'RET'], axis=1).values)
    y_test = test_data['RET'].values

    X_train = torch.tensor(X_train, dtype=torch.float32)
    y_train = torch.tensor(y_train, dtype=torch.float32).view(-1, 1)
    X_test = torch.tensor(X_test, dtype=torch.float32)
    y_test = torch.tensor(y_test, dtype=torch.float32).view(-1, 1)

    patience = 30  # 等待多少个epoch没有改善才停止训练
    best_loss = float('inf')
    patience_counter = 0
    best_model_state = None
    
    # net = NN2(num_inputs,num_hidden,num_output)
    net=Net3()
    optimizer = optim.SGD(net.parameters(), lr=0.03)
    # 训练过程
    for epoch in tqdm(range(num_epochs), desc='Epoch', unit='item',leave=True):
        net.train()  # 设置模型为训练模式
        for X_batch, y_batch in train_loader:
            # 前向传播
            y_pred = net(X_batch)
            
            # 计算损失
            loss = criterion(y_pred, y_batch)
            
            # 清空梯度
            optimizer.zero_grad()
            
            # 反向传播
            loss.backward()
            
            # 更新参数
            optimizer.step()
        
        # 每 10 个 epoch 打印一次损失和 R² 分数
        if (epoch + 1) % 10 == 0:
            net.eval()  # 设置模型为评估模式
            with torch.no_grad():
                # 计算训练集损失和 R² 分数
                y_train_pred = net(X_train)
                train_loss = criterion(y_train_pred, y_train)
                train_r2 = r2_score(y_train.numpy(), y_train_pred.numpy())
                
                # 计算测试集损失和 R² 分数
                y_test_pred = net(X_test)
                test_loss = criterion(y_test_pred, y_test)
                test_r2 = r2_score(y_test.numpy(), y_test_pred.numpy())
    
            print(f'Epoch [{epoch + 1}/{num_epochs}], Train : Loss: {train_loss.item():.4f}, R²: {train_r2:.4f} ||| Test : Loss: {test_loss.item():.4f}, R²: {test_r2:.4f}')
            if test_loss < best_loss:
                best_loss = test_loss
                patience_counter = 0
                # 更新最佳模型状态
                best_model_state = net.state_dict()
            else:
                patience_counter += 10
                if patience_counter >= patience:
                    print(f'Early stopping at epoch {epoch + 1}')
                    break
    # Predict on test data
    if best_model_state is not None:
        net.load_state_dict(best_model_state)
    net.eval()  # 设置模型为评估模式
    with torch.no_grad():
        y_test_pred = net(X_test)
        y_test_pred = net(X_test)
    
    pred_y_list.extend(y_test_pred)
    true_y_list.extend(y_test)

# Convert lists to numpy arrays
pred_y_array = np.array(pred_y_list)
true_y_array = np.array(true_y_list)

# Calculate and print R^2
print('R^2: {:.2f}%'.format(r2_score(true_y_array, pred_y_array)))

# NN3前15年

In [ ]:
num_inputs=920
num_output=1
num_hidden=128
batch_size=1024
num_epochs=60
criterion = nn.MSELoss()
class NN3(nn.Module):
    def __init__(self, num_inputs, num_hidden, num_output):
        super(NN3, self).__init__()
        
        # 定义网络结构
        self.model = nn.Sequential(
            nn.Linear(num_inputs, num_hidden),
            nn.ReLU(),
            # nn.Dropout(0.2), 
            nn.Linear(num_hidden, num_hidden),
            nn.ReLU(),
            # nn.Dropout(0.2), 
            nn.Linear(num_hidden,num_output)
        )
        
        # 初始化权重
        self._initialize_weights()

    def _initialize_weights(self):
        for m in self.model:
            if isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, std=0.01)
    
    def forward(self, x):
        return self.model(x)

In [ ]:
start_year_train = 1957
start_year_test = 2008
end_year_test = 2016

# Convert the 'Date' column to datetime if it's not already
data['DATE'] = pd.to_datetime(data['DATE'], format='%Y%m')

pred_y_list = []
true_y_list = []
net = NN3(num_inputs,num_hidden,num_output)
optimizer = optim.SGD(net.parameters(), lr=0.03)

# Loop over the years
for year in tqdm(range(start_year_test, end_year_test + 1), desc='Processing', unit='item'):
    # Define the start and end dates for training and test
    start_date_train = pd.Timestamp(year=start_year_train, month=1, day=1)
    end_date_train = pd.Timestamp(year=year - 13, month=12, day=31)

    start_date_val = pd.Timestamp(year=year - 12, month=1, day=1)
    end_date_val = pd.Timestamp(year=year - 1, month=12, day=31)

    start_date_test = pd.Timestamp(year=year, month=1, day=1)
    end_date_test = pd.Timestamp(year=year, month=12, day=31)

    print(f'start_date_train: {start_date_train.strftime("%Y-%m")}')
    print(f'end_date_train: {end_date_train.strftime("%Y-%m")}')
    print(f'start_date_val: {start_date_val.strftime("%Y-%m")}')
    print(f'end_date_val: {end_date_val.strftime("%Y-%m")}')
    print(f'start_date_test: {start_date_test.strftime("%Y-%m")}')
    print(f'end_date_test: {end_date_test.strftime("%Y-%m")}')
    
    # Select the data for training and test
    train_data = data[(data['DATE'] >= start_date_train) & (data['DATE'] <= end_date_val)]
    test_data = data[(data['DATE'] >= start_date_test) & (data['DATE'] <= end_date_test)]

    # Extract features and target variables
    X_train = torch.tensor(train_data.drop(['permno', 'DATE', 'RET'], axis=1).values, dtype=torch.float32)
    y_train = torch.tensor(train_data['RET'].values, dtype=torch.float32).view(-1, 1)  
    X_test = torch.tensor(test_data.drop(['permno', 'DATE', 'RET'], axis=1).values, dtype=torch.float32)
    y_test = torch.tensor(test_data['RET'].values, dtype=torch.float32).view(-1, 1)
    
    train_dataset = TensorDataset(X_train, y_train)
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=False)
    test_dataset = TensorDataset(X_test, y_test)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

    patience = 30  # 等待多少个epoch没有改善才停止训练
    best_loss = float('inf')
    patience_counter = 0
    best_model_state = None
    
    # 训练过程
    for epoch in tqdm(range(num_epochs), desc='Epoch', unit='item'):
        net.train()  # 设置模型为训练模式
        for X_batch, y_batch in train_loader:
            # 前向传播
            y_pred = net(X_batch)
            
            # 计算损失
            loss = criterion(y_pred, y_batch)
            
            # 清空梯度
            optimizer.zero_grad()
            
            # 反向传播
            loss.backward()
            
            # 更新参数
            optimizer.step()
        
        # 每 10 个 epoch 打印一次损失和 R² 分数
        if (epoch + 1) % 10 == 0:
            net.eval()  # 设置模型为评估模式
            with torch.no_grad():
                # 计算训练集损失和 R² 分数
                y_train_pred = net(X_train)
                train_loss = criterion(y_train_pred, y_train)
                train_r2 = r2_score(y_train.numpy(), y_train_pred.numpy())
                
                # 计算测试集损失和 R² 分数
                y_test_pred = net(X_test)
                test_loss = criterion(y_test_pred, y_test)
                test_r2 = r2_score(y_test.numpy(), y_test_pred.numpy())
    
            print(f'Epoch [{epoch + 1}/{num_epochs}], Train : Loss: {train_loss.item():.4f}, R²: {train_r2:.4f} ||| Test : Loss: {test_loss.item():.4f}, R²: {test_r2:.4f}')
            if test_loss < best_loss:
                best_loss = test_loss
                patience_counter = 0
                # 更新最佳模型状态
                best_model_state = net.state_dict()
            else:
                patience_counter += 10
                if patience_counter >= patience:
                    print(f'Early stopping at epoch {epoch + 1}')
                    break
    # Predict on test data
    if best_model_state is not None:
        net.load_state_dict(best_model_state)
    net.eval()  # 设置模型为评估模式
    with torch.no_grad():
        y_test_pred = net(X_test)
        y_test_pred = net(X_test)
    
    pred_y_list.extend(y_test_pred)
    true_y_list.extend(y_test)
    tmp=pd.DataFrame({'pred':[i[0] for i in np.array(y_test_pred)],'true':[i[0] for i in np.array(y_test)]})
    tmp.to_csv(f'output/{year}.csv')
    gc.collect()
# Convert lists to numpy arrays
y_preds=[np.array(i) for i in pred_y_list]
y_trues=[np.array(i) for i in true_y_list]

# Calculate and print R^2
print('R^2: {:.2f}%'.format(r2_score(y_trues, y_preds)))

In [ ]:
end=pd.read_csv(r'output/1987.csv')
for i in range(1987,2017):
    tmp=pd.read_csv(rf'output/{str(i)}.csv')
    end=pd.concat([end,tmp],axis=0)
end=end.drop('Unnamed: 0',axis=1)
print('R^2: {:.2f}'.format(r2_score(end['true'].values,end['pred'].values)))

# NN3后十五年

In [ ]:
num_inputs=920
num_output=1
num_hidden=128
batch_size=1024
num_epochs=60
criterion = nn.MSELoss()
class NN3(nn.Module):
    def __init__(self, num_inputs, num_hidden, num_output):
        super(NN3, self).__init__()
        
        # 定义网络结构
        self.model = nn.Sequential(
            nn.Linear(num_inputs, num_hidden),
            nn.ReLU(),
            # nn.Dropout(0.2), 
            nn.Linear(num_hidden, num_hidden),
            nn.ReLU(),
            # nn.Dropout(0.2), 
            nn.Linear(num_hidden,num_output)
        )
        
        # 初始化权重
        self._initialize_weights()

    def _initialize_weights(self):
        for m in self.model:
            if isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, std=0.01)
    
    def forward(self, x):
        return self.model(x)

In [ ]:
start_year_train = 1957
start_year_test = 1996
end_year_test = 2016

# Convert the 'Date' column to datetime if it's not already
data['DATE'] = pd.to_datetime(data['DATE'], format='%Y%m')

pred_y_list = []
true_y_list = []
net = NN3(num_inputs,num_hidden,num_output)
optimizer = optim.SGD(net.parameters(), lr=0.03)

# Loop over the years
for year in tqdm(range(start_year_test, end_year_test + 1), desc='Processing', unit='item'):
    # Define the start and end dates for training and test
    start_date_train = pd.Timestamp(year=start_year_train, month=1, day=1)
    end_date_train = pd.Timestamp(year=year - 13, month=12, day=31)

    start_date_val = pd.Timestamp(year=year - 12, month=1, day=1)
    end_date_val = pd.Timestamp(year=year, month=4, day=30)

    start_date_test = pd.Timestamp(year=year, month=1, day=1)
    end_date_test = pd.Timestamp(year=year, month=12, day=31)

    print(f'start_date_train: {start_date_train.strftime("%Y-%m")}')
    print(f'end_date_train: {end_date_train.strftime("%Y-%m")}')
    print(f'start_date_val: {start_date_val.strftime("%Y-%m")}')
    # print(f'end_date_val: {end_date_val.strftime("%Y-%m")}')
    print(f'end_date_val: {(end_date_val.replace(year=end_date_val.year - 1,month=end_date_val.month+8)).strftime("%Y-%m")}')
    print(f'start_date_test: {start_date_test.strftime("%Y-%m")}')
    print(f'end_date_test: {end_date_test.strftime("%Y-%m")}')
    
    # Select the data for training and test
    train_data = data[(data['DATE'] >= start_date_train) & (data['DATE'] <= end_date_val)]
    test_data = data[(data['DATE'] >= start_date_test) & (data['DATE'] <= end_date_test)]

    # Extract features and target variables
    X_train = torch.tensor(train_data.drop(['permno', 'DATE', 'RET'], axis=1).values, dtype=torch.float32)
    y_train = torch.tensor(train_data['RET'].values, dtype=torch.float32).view(-1, 1)  
    X_test = torch.tensor(test_data.drop(['permno', 'DATE', 'RET'], axis=1).values, dtype=torch.float32)
    y_test = torch.tensor(test_data['RET'].values, dtype=torch.float32).view(-1, 1)
    
    train_dataset = TensorDataset(X_train, y_train)
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=False)
    test_dataset = TensorDataset(X_test, y_test)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

    patience = 30  # 等待多少个epoch没有改善才停止训练
    best_loss = float('inf')
    patience_counter = 0
    best_model_state = None
    
    # 训练过程
    for epoch in tqdm(range(num_epochs), desc='Epoch', unit='item'):
        net.train()  # 设置模型为训练模式
        for X_batch, y_batch in train_loader:
            # 前向传播
            y_pred = net(X_batch)
            # 计算损失
            loss = criterion(y_pred, y_batch)
            # 清空梯度
            optimizer.zero_grad()
            # 反向传播
            loss.backward()
            # 更新参数
            optimizer.step()
        # 每 10 个 epoch 打印一次损失和 R² 分数
        if (epoch + 1) % 10 == 0:
            net.eval()  # 设置模型为评估模式
            with torch.no_grad():
                # 计算训练集损失和 R² 分数
                y_train_pred = net(X_train)
                train_loss = criterion(y_train_pred, y_train)
                train_r2 = r2_score(y_train.numpy(), y_train_pred.numpy())
                
                # 计算测试集损失和 R² 分数
                y_test_pred = net(X_test)
                test_loss = criterion(y_test_pred, y_test)
                test_r2 = r2_score(y_test.numpy(), y_test_pred.numpy())
    
            print(f'Epoch [{epoch + 1}/{num_epochs}], Train : Loss: {train_loss.item():.4f}, R²: {train_r2:.4f} ||| Test : Loss: {test_loss.item():.4f}, R²: {test_r2:.4f}')
            if test_loss < best_loss:
                best_loss = test_loss
                patience_counter = 0
                # 更新最佳模型状态
                best_model_state = net.state_dict()
            else:
                patience_counter += 10
                if patience_counter >= patience:
                    print(f'Early stopping at epoch {epoch + 1}')
                    break
    # Predict on test data
    if best_model_state is not None:
        net.load_state_dict(best_model_state)
    net.eval()  # 设置模型为评估模式
    with torch.no_grad():
        y_test_pred = net(X_test)
        y_test_pred = net(X_test)
    
    pred_y_list.extend(y_test_pred)
    true_y_list.extend(y_test)
    tmp=pd.DataFrame({'pred':[i[0] for i in np.array(y_test_pred)],'true':[i[0] for i in np.array(y_test)]})
    tmp.to_csv(f'six/{year}.csv')
    gc.collect()
# Convert lists to numpy arrays
y_preds=[np.array(i) for i in pred_y_list]
y_trues=[np.array(i) for i in true_y_list]

# Calculate and print R^2
print('R^2: {:.2f}%'.format(r2_score(y_trues, y_preds)))

In [ ]:
end=pd.read_csv(r'six/1987.csv')
for i in range(1987,2017):
    tmp=pd.read_csv(rf'six/{str(i)}.csv')
    end=pd.concat([end,tmp],axis=0)
end=end.drop('Unnamed: 0',axis=1)
print('R^2: {:.2f}'.format(r2_score(end['true'].values,end['pred'].values)))

# NN5

In [ ]:
num_inputs=920
num_output=1
num_hidden=128
batch_size=1024
num_epochs=60
criterion = nn.MSELoss()
class NN5(nn.Module):
    def __init__(self, num_inputs, num_hidden, num_output):
        super(NN5, self).__init__()
        
        # 定义网络结构
        self.model = nn.Sequential(
            nn.Linear(num_inputs, num_hidden),
            nn.ReLU(),
            nn.Linear(num_hidden, num_hidden),
            nn.ReLU(),
            nn.Linear(num_hidden,num_hidden),
            nn.ReLU(),
            nn.Linear(num_hidden,num_hidden),
            nn.ReLU(),
            nn.Linear(num_hidden,num_output)
        )
        
        # 初始化权重
        self._initialize_weights()

    def _initialize_weights(self):
        for m in self.model:
            if isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, std=0.01)
    
    def forward(self, x):
        return self.model(x)

In [ ]:
start_year_train = 1957
start_year_test = 1987
end_year_test = 2016

# Convert the 'Date' column to datetime if it's not already
data['DATE'] = pd.to_datetime(data['DATE'], format='%Y%m')

pred_y_list = []
true_y_list = []
net = NN5(num_inputs,num_hidden,num_output)
optimizer = optim.SGD(net.parameters(), lr=0.03)

# Loop over the years
for year in tqdm(range(start_year_test, end_year_test + 1), desc='Processing', unit='item'):
    # Define the start and end dates for training and test
    start_date_train = pd.Timestamp(year=start_year_train, month=1, day=1)
    end_date_train = pd.Timestamp(year=year - 13, month=12, day=31)

    start_date_val = pd.Timestamp(year=year - 12, month=1, day=1)
    end_date_val = pd.Timestamp(year=year - 1, month=12, day=31)

    start_date_test = pd.Timestamp(year=year, month=1, day=1)
    end_date_test = pd.Timestamp(year=year, month=12, day=31)

    print(f'start_date_train: {start_date_train.strftime("%Y-%m")}')
    print(f'end_date_train: {end_date_train.strftime("%Y-%m")}')
    print(f'start_date_val: {start_date_val.strftime("%Y-%m")}')
    print(f'end_date_val: {end_date_val.strftime("%Y-%m")}')
    print(f'start_date_test: {start_date_test.strftime("%Y-%m")}')
    print(f'end_date_test: {end_date_test.strftime("%Y-%m")}')
    
    # Select the data for training and test
    train_data = data[(data['DATE'] >= start_date_train) & (data['DATE'] <= end_date_val)]
    test_data = data[(data['DATE'] >= start_date_test) & (data['DATE'] <= end_date_test)]

    # Extract features and target variables
    X_train = torch.tensor(train_data.drop(['permno', 'DATE', 'RET'], axis=1).values, dtype=torch.float32)
    y_train = torch.tensor(train_data['RET'].values, dtype=torch.float32).view(-1, 1)  
    X_test = torch.tensor(test_data.drop(['permno', 'DATE', 'RET'], axis=1).values, dtype=torch.float32)
    y_test = torch.tensor(test_data['RET'].values, dtype=torch.float32).view(-1, 1)
    
    train_dataset = TensorDataset(X_train, y_train)
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=False)
    test_dataset = TensorDataset(X_test, y_test)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

    patience = 30  # 等待多少个epoch没有改善才停止训练
    best_loss = float('inf')
    patience_counter = 0
    best_model_state = None
    
    # 训练过程
    for epoch in tqdm(range(num_epochs), desc='Epoch', unit='item'):
        net.train()  # 设置模型为训练模式
        for X_batch, y_batch in train_loader:
            # 前向传播
            y_pred = net(X_batch)
            # 计算损失
            loss = criterion(y_pred, y_batch)
            # 清空梯度
            optimizer.zero_grad()
            # 反向传播
            loss.backward()
            # 更新参数
            optimizer.step()
        # 每 10 个 epoch 打印一次损失和 R² 分数
        if (epoch + 1) % 10 == 0:
            net.eval()  # 设置模型为评估模式
            with torch.no_grad():
                # 计算训练集损失和 R² 分数
                y_train_pred = net(X_train)
                train_loss = criterion(y_train_pred, y_train)
                train_r2 = r2_score(y_train.numpy(), y_train_pred.numpy())
                
                # 计算测试集损失和 R² 分数
                y_test_pred = net(X_test)
                test_loss = criterion(y_test_pred, y_test)
                test_r2 = r2_score(y_test.numpy(), y_test_pred.numpy())
    
            print(f'Epoch [{epoch + 1}/{num_epochs}], Train : Loss: {train_loss.item():.4f}, R²: {train_r2:.4f} ||| Test : Loss: {test_loss.item():.4f}, R²: {test_r2:.4f}')
            if test_loss < best_loss:
                best_loss = test_loss
                patience_counter = 0
                # 更新最佳模型状态
                best_model_state = net.state_dict()
            else:
                patience_counter += 10
                if patience_counter >= patience:
                    print(f'Early stopping at epoch {epoch + 1}')
                    break
    # Predict on test data
    if best_model_state is not None:
        net.load_state_dict(best_model_state)
    net.eval()  # 设置模型为评估模式
    with torch.no_grad():
        y_test_pred = net(X_test)
        y_test_pred = net(X_test)
    
    pred_y_list.extend(y_test_pred)
    true_y_list.extend(y_test)
    tmp=pd.DataFrame({'pred':[i[0] for i in np.array(y_test_pred)],'true':[i[0] for i in np.array(y_test)]})
    tmp.to_csv(f'NN5/{year}.csv')
    gc.collect()
# Convert lists to numpy arrays
y_preds=[np.array(i) for i in pred_y_list]
y_trues=[np.array(i) for i in true_y_list]

# Calculate and print R^2
print('R^2: {:.2f}%'.format(r2_score(y_trues, y_preds)))

# Single

In [ ]:
data['DATE'] = pd.to_datetime(data['DATE'], format='%Y%m')
data0=data[data['permno']==21573].reset_index(drop=True)

In [ ]:
num_inputs=920
num_output=1
num_hidden=128
batch_size=1024
num_epochs=60
criterion = nn.MSELoss()
class NN5(nn.Module):
    def __init__(self, num_inputs, num_hidden, num_output):
        super(NN5, self).__init__()
        
        # 定义网络结构
        self.model = nn.Sequential(
            nn.Linear(num_inputs, num_hidden),
            nn.ReLU(),
            nn.Linear(num_hidden, num_hidden),
            nn.ReLU(),
            nn.Linear(num_hidden,num_hidden),
            nn.ReLU(),
            nn.Linear(num_hidden,num_hidden),
            nn.ReLU(),
            nn.Linear(num_hidden,num_output)
        )
        
        # 初始化权重
        self._initialize_weights()

    def _initialize_weights(self):
        for m in self.model:
            if isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, std=0.01)
    
    def forward(self, x):
        return self.model(x)

In [ ]:
start_year_train=int(data0['DATE'].min().year)
start_year_test=int(int(data0['DATE'].min().year)+int(data0['DATE'].max().year-data0['DATE'].min().year)*0.75)
end_year_test=int(data0['DATE'].max().year)

In [ ]:
net = NN5(num_inputs,num_hidden,num_output)
optimizer = optim.SGD(net.parameters(), lr=0.03)
pred_y_list = []
true_y_list = []

# Loop over the years
for year in tqdm(range(start_year_test, end_year_test + 1), desc='Processing', unit='item'):
    # Define the start and end dates for training and test
    start_date_train = pd.Timestamp(year=start_year_train, month=1, day=1)
    end_date_train = pd.Timestamp(year=year - 13, month=12, day=31)

    start_date_val = pd.Timestamp(year=year - 12, month=1, day=1)
    end_date_val = pd.Timestamp(year=year - 1, month=12, day=31)

    start_date_test = pd.Timestamp(year=year, month=1, day=1)
    end_date_test = pd.Timestamp(year=year, month=12, day=31)

    print(f'start_date_train: {start_date_train.strftime("%Y-%m")}')
    print(f'end_date_train: {end_date_train.strftime("%Y-%m")}')
    print(f'start_date_val: {start_date_val.strftime("%Y-%m")}')
    print(f'end_date_val: {end_date_val.strftime("%Y-%m")}')
    print(f'start_date_test: {start_date_test.strftime("%Y-%m")}')
    print(f'end_date_test: {end_date_test.strftime("%Y-%m")}')
    
    # Select the data for training and test
    train_data = data0[(data0['DATE'] >= start_date_train) & (data0['DATE'] <= end_date_val)]
    test_data = data0[(data0['DATE'] >= start_date_test) & (data0['DATE'] <= end_date_test)]

    # Extract features and target variables
    X_train = torch.tensor(train_data.drop(['permno', 'DATE', 'RET'], axis=1).values, dtype=torch.float32)
    y_train = torch.tensor(train_data['RET'].values, dtype=torch.float32).view(-1, 1)  
    X_test = torch.tensor(test_data.drop(['permno', 'DATE', 'RET'], axis=1).values, dtype=torch.float32)
    y_test = torch.tensor(test_data['RET'].values, dtype=torch.float32).view(-1, 1)
    
    train_dataset = TensorDataset(X_train, y_train)
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=False)
    test_dataset = TensorDataset(X_test, y_test)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

    patience = 30  # 等待多少个epoch没有改善才停止训练
    best_loss = float('inf')
    patience_counter = 0
    best_model_state = None
    
    # 训练过程
    for epoch in tqdm(range(num_epochs), desc='Epoch', unit='item'):
        net.train()  # 设置模型为训练模式
        for X_batch, y_batch in train_loader:
            # 前向传播
            y_pred = net(X_batch)
            print(X_batch.shape,y_batch.shape,y_pred.shape)
            break
            # 计算损失
            loss = criterion(y_pred, y_batch)
            
            # 清空梯度
            optimizer.zero_grad()
            
            # 反向传播
            loss.backward()
            
            # 更新参数
            optimizer.step()
        
        # 每 10 个 epoch 打印一次损失和 R² 分数
        if (epoch + 1) % 10 == 0:
            net.eval()  # 设置模型为评估模式
            with torch.no_grad():
                # 计算训练集损失和 R² 分数
                y_train_pred = net(X_train)
                train_loss = criterion(y_train_pred, y_train)
                train_r2 = r2_score(y_train.numpy(), y_train_pred.numpy())
                
                # 计算测试集损失和 R² 分数
                y_test_pred = net(X_test)
                test_loss = criterion(y_test_pred, y_test)
                test_r2 = r2_score(y_test.numpy(), y_test_pred.numpy())
    
            print(f'Epoch [{epoch + 1}/{num_epochs}], Train : Loss: {train_loss.item():.4f}, R²: {train_r2:.4f} ||| Test : Loss: {test_loss.item():.4f}, R²: {test_r2:.4f}')
            if test_loss < best_loss:
                best_loss = test_loss
                patience_counter = 0
                # 更新最佳模型状态
                best_model_state = net.state_dict()
            else:
                patience_counter += 10
                if patience_counter >= patience:
                    print(f'Early stopping at epoch {epoch + 1}')
                    break
    # Predict on test data
    if best_model_state is not None:
        net.load_state_dict(best_model_state)
    net.eval()  # 设置模型为评估模式
    with torch.no_grad():
        y_test_pred = net(X_test)
        y_test_pred = net(X_test)
    
    pred_y_list.extend(y_test_pred)
    true_y_list.extend(y_test)

# Convert lists to numpy arrays
y_preds=[np.array(i) for i in pred_y_list]
y_trues=[np.array(i) for i in true_y_list]

# Calculate and print R^2
print('R^2: {:.2f}%'.format(r2_score(y_trues,y_preds)))

# LSTM

In [ ]:
parser = argparse.ArgumentParser()

parser.add_argument('--corpusFile', default='work/raw_data/data/characteristics_with_ind_interaction_return22222.pkl')

# TODO 常改动参数
parser.add_argument('--gpu', default=0, type=int) # gpu 卡号
parser.add_argument('--epochs', default=100, type=int) # 训练轮数
parser.add_argument('--layers', default=2, type=int) # LSTM层数
parser.add_argument('--input_size', default=920, type=int) #输入特征的维度
parser.add_argument('--hidden_size', default=256, type=int) #隐藏层的维度
parser.add_argument('--lr', default=0.01, type=float) #learning rate 学习率
parser.add_argument('--sequence_length', default=20, type=int) # sequence的长度，默认是用前五天的数据来预测下一天的收盘价
parser.add_argument('--batch_size', default=256, type=int)
parser.add_argument('--useGPU', default=False, type=bool) #是否使用GPU
parser.add_argument('--batch_first', default=True, type=bool) #是否将batch_size放在第一维
parser.add_argument('--dropout', default=0.1, type=float)
parser.add_argument('--save_file', default='model/stock.pkl') # 模型保存位置

args, unknown = parser.parse_known_args()

device = torch.device(f"cuda:{args.gpu}" if torch.cuda.is_available() and args.useGPU else "cpu")
args.device = device

In [ ]:
class lstm(nn.Module):
    def __init__(self, input_size=8, hidden_size=32, num_layers=1 , output_size=1 , dropout=0, batch_first=True):
        super(lstm, self).__init__()
        # lstm的输入 #batch,seq_len, input_size
        self.hidden_size = hidden_size
        self.input_size = input_size
        self.num_layers = num_layers
        self.output_size = output_size
        self.dropout = dropout
        self.batch_first = batch_first
        self.rnn = nn.LSTM(input_size=self.input_size, hidden_size=self.hidden_size, num_layers=self.num_layers, batch_first=self.batch_first, dropout=self.dropout )
        self.linear = nn.Linear(self.hidden_size, self.output_size)

    def forward(self, x):
        out, (hidden, cell) = self.rnn(x)  # x.shape : batch, seq_len, hidden_size , hn.shape and cn.shape : num_layes * direction_numbers, batch, hidden_size
        # a, b, c = hidden.shape
        # out = self.linear(hidden.reshape(a * b, c))
        out = self.linear(hidden)
        return out

In [ ]:
data['DATE'] = pd.to_datetime(data['DATE'], format='%Y%m')
data0=data[data['permno']==21573].reset_index(drop=True)

In [ ]:
start_year_train=int(data0['DATE'].min().year)
start_year_test=int(int(data0['DATE'].min().year)+int(data0['DATE'].max().year-data0['DATE'].min().year)*0.75)
end_year_test=int(data0['DATE'].max().year)

In [ ]:
def create_sequences(data, seq_length):
    X = []
    Y = []
    XX = data.drop(['permno', 'DATE', 'RET'], axis=1).values
    yy = data['RET'].values
    for i in range(len(data) - seq_length):
        X.append(XX[i:(i + seq_length), :])
        Y.append(yy[i + seq_length])
    return torch.tensor(X, dtype=torch.float32), torch.tensor(Y, dtype=torch.float32)
model = lstm(input_size=args.input_size, hidden_size=args.hidden_size, num_layers=args.layers , output_size=1, dropout=args.dropout, batch_first=args.batch_first )
model.to(args.device)
criterion = nn.MSELoss()  # 定义损失函数
optimizer = torch.optim.Adam(model.parameters(), lr=args.lr)  # Adam梯度下降  学习率=0.001
pred_y_list = []
true_y_list = []

# Loop over the years
for year in tqdm(range(start_year_test, end_year_test + 1), desc='Processing', unit='item'):
    # Define the start and end dates for training and test
    start_date_train = pd.Timestamp(year=start_year_train, month=1, day=1)
    end_date_train = pd.Timestamp(year=year - 13, month=12, day=31)

    start_date_val = pd.Timestamp(year=year - 12, month=1, day=1)
    end_date_val = pd.Timestamp(year=year - 1, month=12, day=31)

    start_date_test = pd.Timestamp(year=year, month=1, day=1)
    end_date_test = pd.Timestamp(year=year, month=12, day=31)

    print(f'start_date_train: {start_date_train.strftime("%Y-%m")}')
    print(f'end_date_train: {end_date_train.strftime("%Y-%m")}')
    print(f'start_date_val: {start_date_val.strftime("%Y-%m")}')
    print(f'end_date_val: {end_date_val.strftime("%Y-%m")}')
    print(f'start_date_test: {start_date_test.strftime("%Y-%m")}')
    print(f'end_date_test: {end_date_test.strftime("%Y-%m")}')
    
    # Select the data for training and test
    train_data = data0[(data0['DATE'] >= start_date_train) & (data0['DATE'] <= end_date_val)]
    test_data = data0[(data0['DATE'] >= start_date_test) & (data0['DATE'] <= end_date_test)]
    
    sequence = args.sequence_length
    
    # 构建训练和测试的时间序列
    train_sequences, train_labels = create_sequences(train_data, sequence)
    test_sequences, test_labels = create_sequences(test_data, sequence)
    
    train_dataset = TensorDataset(train_sequences, train_labels)
    train_loader = DataLoader(train_dataset, batch_size=args.batch_size, shuffle=False)
    test_dataset = TensorDataset(test_sequences, test_labels)
    test_loader = DataLoader(test_dataset, batch_size=args.batch_size, shuffle=False)
    
    # for batch in train_loader:
    #     print(f"Batch X shape: {batch[0].shape}")
    #     print(f"Batch y shape: {batch[1].shape}")
    #     break
    
    # patience = 30  # 等待多少个epoch没有改善才停止训练
    # best_loss = float('inf')
    # patience_counter = 0
    # best_model_state = None
    
    # 训练过程
    for epoch in tqdm(range(args.epochs), desc='Epoch', unit='item'):
        model.train()
        train_loss = 0
        for idx, (data, label) in enumerate(train_loader):
            if args.useGPU:
                data1 = data.squeeze(1).cuda()
                pred = model(Variable(data1).cuda())
                # print(pred.shape)
                pred = pred[1,:,:]
                label = label.unsqueeze(1).cuda()
                # print(label.shape)
            else:
                data1 = data.squeeze(1)
                pred = model(Variable(data1))
                pred = pred[1, :, :]
                label = label.unsqueeze(1)
            train_r2 = r2_score(label.detach(), pred.detach())
            loss = criterion(pred, label)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
        print(f'train_loss:{train_loss}')
        print(f'train_r2:{train_r2}')
        print('-------------------------------Testing-----------------------------')
        model.eval()  # 设置模型为评估模式
        test_loss=0
        for idx, (x, label) in enumerate(test_loader):
            if args.useGPU:
                x = x.squeeze(1).cuda()  # batch_size,seq_len,input_size
            else:
                x = x.squeeze(1)
                pred = model(x)
                list = pred.data.squeeze(1).tolist()
                preds.extend(list[-1])
                labels.extend(label.tolist())
                print('你的后面我在呢')
            loss = criterion(pred, label)
            test_loss += loss.item()
            test_r2 = r2_score(labels, preds)
            print(f'test_loss:{test_loss}')
            print(f'test_r2:{test_r2}')
            
        #print(f'Epoch [{epoch + 1}/{num_epochs}], Train : Loss: {train_loss.item():.4f}, R²: {train_r2:.4f} ||| Test : Loss: {test_loss.item():.4f}, R²: {test_r2:.4f}')


        
#         model.train()  # 设置模型为训练模式
#         for X_batch, y_batch in train_loader:
#             # 前向传播
#             y_pred = net(X_batch)
            
#             # 计算损失
#             loss = criterion(y_pred, y_batch)
            
#             # 清空梯度
#             optimizer.zero_grad()
            
#             # 反向传播
#             loss.backward()
            
#             # 更新参数
#             optimizer.step()
        
#         # 每 10 个 epoch 打印一次损失和 R² 分数
#         if (epoch + 1) % 10 == 0:
#             net.eval()  # 设置模型为评估模式
#             with torch.no_grad():
#                 # 计算训练集损失和 R² 分数
#                 y_train_pred = net(X_train)
#                 train_loss = criterion(y_train_pred, y_train)
#                 train_r2 = r2_score(y_train.numpy(), y_train_pred.numpy())
                
#                 # 计算测试集损失和 R² 分数
#                 y_test_pred = net(X_test)
#                 test_loss = criterion(y_test_pred, y_test)
#                 test_r2 = r2_score(y_test.numpy(), y_test_pred.numpy())
    
#             print(f'Epoch [{epoch + 1}/{num_epochs}], Train : Loss: {train_loss.item():.4f}, R²: {train_r2:.4f} ||| Test : Loss: {test_loss.item():.4f}, R²: {test_r2:.4f}')
#             if test_loss < best_loss:
#                 best_loss = test_loss
#                 patience_counter = 0
#                 # 更新最佳模型状态
#                 best_model_state = net.state_dict()
#             else:
#                 patience_counter += 10
#                 if patience_counter >= patience:
#                     print(f'Early stopping at epoch {epoch + 1}')
#                     break
#     # Predict on test data
#     if best_model_state is not None:
#         net.load_state_dict(best_model_state)
#     net.eval()  # 设置模型为评估模式
#     with torch.no_grad():
#         y_test_pred = net(X_test)
#         y_test_pred = net(X_test)
    
#     pred_y_list.extend(y_test_pred)
#     true_y_list.extend(y_test)

# # Convert lists to numpy arrays
# y_preds=[np.array(i) for i in pred_y_list]
# y_trues=[np.array(i) for i in true_y_list]

# # Calculate and print R^2
# print('R^2: {:.2f}%'.format(r2_score(y_trues,y_preds)))

In [ ]:
def train():
    model = lstm(input_size=args.input_size, hidden_size=args.hidden_size, num_layers=args.layers , output_size=1, dropout=args.dropout, batch_first=args.batch_first )
    model.to(args.device)
    criterion = nn.MSELoss()  # 定义损失函数
    optimizer = torch.optim.Adam(model.parameters(), lr=args.lr)  # Adam梯度下降  学习率=0.001

    close_max, close_min, train_loader, test_loader = getData(args.corpusFile,args.sequence_length,args.batch_size )
    for i in range(args.epochs):
        total_loss = 0
        for idx, (data, label) in enumerate(train_loader):
            if args.useGPU:
                data1 = data.squeeze(1).cuda()
                pred = model(Variable(data1).cuda())
                # print(pred.shape)
                pred = pred[1,:,:]
                label = label.unsqueeze(1).cuda()
                # print(label.shape)
            else:
                data1 = data.squeeze(1)
                pred = model(Variable(data1))
                pred = pred[1, :, :]
                label = label.unsqueeze(1)
            loss = criterion(pred, label)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        print(total_loss)
        if i % 10 == 0:
            # torch.save(model, args.save_file)
            torch.save({'state_dict': model.state_dict()}, args.save_file)
            print('第%d epoch，保存模型' % i)
    # torch.save(model, args.save_file)
    torch.save({'state_dict': model.state_dict()}, args.save_file)

train()

In [ ]:
def eval():
    # model = torch.load(args.save_file)
    model = lstm(input_size=args.input_size, hidden_size=args.hidden_size, num_layers=args.layers , output_size=1)
    model.to(args.device)
    checkpoint = torch.load(args.save_file)
    model.load_state_dict(checkpoint['state_dict'])
    preds = []
    labels = []
    close_max, close_min, train_loader, test_loader = getData(args.corpusFile, args.sequence_length, args.batch_size)
    for idx, (x, label) in enumerate(test_loader):
        if args.useGPU:
            x = x.squeeze(1).cuda()  # batch_size,seq_len,input_size
        else:
            x = x.squeeze(1)
        pred = model(x)
        list = pred.data.squeeze(1).tolist()
        preds.extend(list[-1])
        labels.extend(label.tolist())

    for i in range(len(preds)):
        print('预测值是%.2f,真实值是%.2f' % (
        preds[i][0] * (close_max - close_min) + close_min, labels[i] * (close_max - close_min) + close_min))